# Load hiring data → Delta (daily)

Reads the committed `public/data/*.json` (produced by the GitHub Action) and updates the `frontier_labs` hiring tables. Idempotent — safe to re-run.

- `hiring_jobs` — **overwrite** (ledger tags/lifecycle ⨝ jobs.json rich fields). The ledger is cumulative (keeps closed jobs), so a full rebuild each run is correct.
- `hiring_weekly` / `hiring_weekly_breakdown` — **overwrite** (recomputed from weekly_trends.json).
- `hiring_snapshots` — **append** one dated batch, keyed on the scrape date (skips if that date is already loaded).

**Company names are normalized to the parent level** (Amazon AGI → Amazon, Microsoft Research → Microsoft) and **Mistral is dropped** (not scraping properly). Both are applied on every load — see the normalization cell.

Two extra cells at the bottom: **backfill** a past scrape's snapshot (Aug 5, then Aug 11), and a **one-time cleanup** of existing tables.

**Workflow:** the Action commits fresh JSON → Pull the Git folder (or a Git-sourced job auto-pulls) → run this.

In [ ]:
# --- config ---
CATALOG = "fso_market_intelligence"
SCHEMA  = "frontier_labs"

# Auto-locate public/data in the workspace (so a Git-sourced job needs no path edits)
import subprocess, os
_hits = subprocess.run(["find", "/Workspace", "-maxdepth", "9", "-path", "*/public/data/job_ledger.json"],
                       capture_output=True, text=True).stdout.strip().splitlines()
assert _hits, "job_ledger.json not found under /Workspace — pull the Git folder first"
DATA = os.path.dirname(_hits[0])
print("DATA =", DATA)

In [ ]:
import json
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType, BooleanType, LongType)

## Company normalization — parent-level names + drop Mistral

`remap_exclude(df)` drops excluded companies and renames the rest to the parent level. Applied to
every table below so a full reload always produces clean names and no Mistral. To re-enable Mistral
or tweak a name, edit the two dicts here.

In [ ]:
# Rename to the parent level (strip 'AGI' / 'Research'). These are the only two scraped companies
# with an additive; everything else scraped is already parent-level and kept as-is:
# Anthropic, Google, xAI, OpenAI, Cohere, Moonshot AI, NVIDIA, ByteDance, Tencent.
COMPANY_RENAME = {
    "Amazon AGI":         "Amazon",
    "Microsoft Research": "Microsoft",
}
# Dropped from all tables for now (scraper not producing clean data).
EXCLUDE_COMPANIES = ["Mistral AI"]

def remap_exclude(df):
    """Drop excluded companies, then rename survivors to the parent level."""
    df = df.filter(~F.col("company").isin(EXCLUDE_COMPANIES))
    pairs = []
    for k, v in COMPANY_RENAME.items():
        pairs += [F.lit(k), F.lit(v)]
    m = F.create_map(*pairs)
    return df.withColumn("company", F.coalesce(m[F.col("company")], F.col("company")))

## 1. `hiring_jobs` — ledger (tags + lifecycle) ⨝ postings (title/location/url/description)

In [ ]:
ledger_schema = StructType([
    StructField("job_id", StringType()), StructField("first_seen", StringType()),
    StructField("last_seen", StringType()), StructField("company", StringType()),
    StructField("category", StringType()), StructField("sub_area", StringType()),
    StructField("theme", StringType()), StructField("vertical", StringType()),
    StructField("social_impact", BooleanType()), StructField("active", BooleanType()),
])
with open(f"{DATA}/job_ledger.json") as f:
    led = json.load(f)
ledger_df = (spark.createDataFrame([{"job_id": k, **v} for k, v in led.items()], schema=ledger_schema)
    .withColumn("first_seen", F.to_date("first_seen")).withColumn("last_seen", F.to_date("last_seen")))

posts_schema = StructType([
    StructField("id", StringType()), StructField("title", StringType()),
    StructField("department", StringType()), StructField("location", StringType()),
    StructField("url", StringType()), StructField("source", StringType()),
    StructField("description", StringType()),
])
with open(f"{DATA}/jobs.json") as f:
    jobs_doc = json.load(f)
posts = [{k: j.get(k) for k in ("id", "title", "department", "location", "url", "source", "description")} for j in jobs_doc["jobs"]]
posts_df = spark.createDataFrame(posts, schema=posts_schema).withColumnRenamed("id", "job_id")

hiring = (ledger_df.join(posts_df, "job_id", "left").withColumn("captured_at", F.current_date()))
hiring = remap_exclude(hiring)   # parent-level names + drop Mistral
(hiring.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{SCHEMA}.hiring_jobs"))
print("hiring_jobs rows:", spark.table(f"{CATALOG}.{SCHEMA}.hiring_jobs").count())

## 2. `hiring_weekly` + `hiring_weekly_breakdown` (recomputed from weekly_trends.json)

In [ ]:
with open(f"{DATA}/weekly_trends.json") as f:
    weeks = json.load(f)["weeks"]

wk_rows = [{"week": wk["week"], "company": c, "baseline": bool(wk.get("baseline", False)),
            "total": d.get("total"), "new": d.get("new"), "removed": d.get("removed"),
            "social_impact": d.get("social_impact"), "new_social_impact": d.get("new_social_impact")}
           for wk in weeks for c, d in wk["by_company"].items()]
weekly_schema = StructType([
    StructField("week", StringType()), StructField("company", StringType()), StructField("baseline", BooleanType()),
    StructField("total", LongType()), StructField("new", LongType()), StructField("removed", LongType()),
    StructField("social_impact", LongType()), StructField("new_social_impact", LongType())])
(remap_exclude(spark.createDataFrame(wk_rows, schema=weekly_schema).withColumn("week", F.to_date("week")))
    .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{SCHEMA}.hiring_weekly"))

br_rows = [{"week": wk["week"], "company": c, "dimension": dim, "value": val, "count": cnt}
           for wk in weeks for c, d in wk["by_company"].items()
           for dim, kv in (d.get("dist") or {}).items() for val, cnt in kv.items()]
br_schema = StructType([
    StructField("week", StringType()), StructField("company", StringType()),
    StructField("dimension", StringType()), StructField("value", StringType()), StructField("count", LongType())])
(remap_exclude(spark.createDataFrame(br_rows, schema=br_schema).withColumn("week", F.to_date("week")))
    .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{SCHEMA}.hiring_weekly_breakdown"))

print("hiring_weekly:", spark.table(f"{CATALOG}.{SCHEMA}.hiring_weekly").count(),
      "| breakdown:", spark.table(f"{CATALOG}.{SCHEMA}.hiring_weekly_breakdown").count())

## 3. `hiring_snapshots` — append one dated batch (keyed on the scrape date, idempotent)

In [ ]:
# scrape date from jobs.json scraped_at (the real data day, not the run day)
snap_date = (jobs_doc.get("scraped_at") or "")[:10]
print("scrape date:", snap_date)

snap = f"{CATALOG}.{SCHEMA}.hiring_snapshots"
already = spark.table(snap).where(F.col("snapshot_date") == F.lit(snap_date)).limit(1).count() if snap_date else 1
if snap_date and already == 0:
    (spark.table(f"{CATALOG}.{SCHEMA}.hiring_jobs")
        .select("job_id", "company", "location", "category", "sub_area", "theme", "vertical", "social_impact", "active")
        .withColumn("snapshot_date", F.to_date(F.lit(snap_date)))
        .write.mode("append").saveAsTable(snap))
    print(f"appended snapshot for {snap_date}")
else:
    print(f"snapshot for {snap_date} already present — skipped")
print("hiring_snapshots rows:", spark.table(snap).count(),
      "| distinct dates:", spark.table(snap).select("snapshot_date").distinct().count())

## Backfill a past scrape's snapshot

Adds the `hiring_snapshots` batch for an **older** scrape that was never loaded, by reading that
commit's `job_ledger.json` + `jobs.json` straight from GitHub (no Git-folder switching). It only
touches `hiring_snapshots` — it does **not** overwrite `hiring_jobs`/weekly, so it's safe to run
before or after the main load.

**Run it once per date:** set `BACKFILL_REF` to Aug 5, run → then change to Aug 11, run again.
(Requires cells 1–3 to have run: `CATALOG`/`SCHEMA`, imports, and `remap_exclude`.)

In [ ]:
import urllib.request

REPO = "alym00sa-dev/ai-market-intelligence-public"
# Last commit of each unloaded run:
#   Aug 5  → 3827732      Aug 11 → a55a475
BACKFILL_REF = "3827732"   # <-- Aug 5; change to "a55a475" for Aug 11, then re-run this cell

def _raw(fn):
    url = f"https://raw.githubusercontent.com/{REPO}/{BACKFILL_REF}/public/data/{fn}"
    return json.loads(urllib.request.urlopen(url, timeout=60).read())

bf_led  = _raw("job_ledger.json")
bf_jobs = _raw("jobs.json")
bf_date = (bf_jobs.get("scraped_at") or "")[:10]
print("backfill ref:", BACKFILL_REF, "| scrape date:", bf_date)

_led_schema = StructType([
    StructField("job_id", StringType()), StructField("first_seen", StringType()),
    StructField("last_seen", StringType()), StructField("company", StringType()),
    StructField("category", StringType()), StructField("sub_area", StringType()),
    StructField("theme", StringType()), StructField("vertical", StringType()),
    StructField("social_impact", BooleanType()), StructField("active", BooleanType()),
])
bf_ledger = spark.createDataFrame([{"job_id": k, **v} for k, v in bf_led.items()], schema=_led_schema)
bf_loc = spark.createDataFrame(
    [{"job_id": j.get("id"), "location": j.get("location")} for j in bf_jobs["jobs"]],
    StructType([StructField("job_id", StringType()), StructField("location", StringType())]))
bf = remap_exclude(bf_ledger.join(bf_loc, "job_id", "left"))   # parent-level names + drop Mistral

snap = f"{CATALOG}.{SCHEMA}.hiring_snapshots"
already = spark.table(snap).where(F.col("snapshot_date") == F.lit(bf_date)).limit(1).count() if bf_date else 1
if bf_date and already == 0:
    (bf.select("job_id", "company", "location", "category", "sub_area", "theme", "vertical", "social_impact", "active")
        .withColumn("snapshot_date", F.to_date(F.lit(bf_date)))
        .write.mode("append").saveAsTable(snap))
    print(f"appended backfill snapshot for {bf_date}")
else:
    print(f"snapshot for {bf_date} already present — skipped")
print("distinct snapshot dates now:",
      sorted(r[0].isoformat() for r in spark.table(snap).select("snapshot_date").distinct().collect()))

## One-time cleanup of existing tables (rename + drop Mistral)

The main cells above already produce clean names on overwrite, but `hiring_snapshots` is append-only,
so its historical rows still have old names / Mistral. Run this **once** to fix every existing table
in place. Safe to re-run (idempotent).

In [ ]:
for tbl in ["hiring_jobs", "hiring_snapshots", "hiring_weekly", "hiring_weekly_breakdown"]:
    full = f"{CATALOG}.{SCHEMA}.{tbl}"
    for old, new in COMPANY_RENAME.items():
        spark.sql(f"UPDATE {full} SET company = '{new}' WHERE company = '{old}'")
    for ex in EXCLUDE_COMPANIES:
        spark.sql(f"DELETE FROM {full} WHERE company = '{ex}'")
    print("cleaned", tbl, "— companies:",
          sorted(r[0] for r in spark.table(full).select("company").distinct().collect()))